# V5 Training — FIXED (games ×3 + 6M puzzles + draws + resigns)

**Two critical fixes:**
1. **Cell 5** auto-fetches the Witty shards from `GM-games-with-eval` → `chess-data` (the previous notebook tried to add them but they weren't on `chess-data` yet).
2. **Cell 6** rebuilds `index.json` from `sync.list_data()` — the *actual* files on HF — not from the old index. Previous version emptied the index because it read only puzzle entries.

**Setup:**
- Accelerator: **GPU T4 ×2**
- Secret: **HF_TOKEN**
- Dataset: **omarcodeplay/gm-dataset**

In [ ]:
# ═══ Cell 1 — Bootstrap ═══
import os, shutil, pathlib, sys

SRC_IN = '/kaggle/input/datasets/omarcodeplay/gm-dataset/src'
SRC_OUT = '/kaggle/working/src'
os.makedirs(SRC_OUT, exist_ok=True)

if not os.path.isdir(SRC_IN):
    import glob
    candidates = glob.glob('/kaggle/input/**/src', recursive=True)
    if candidates:
        SRC_IN = candidates[0]
    else:
        raise SystemExit(f'Source not found. Add omarcodeplay/gm-dataset.')

copied = []
for f in pathlib.Path(SRC_IN).glob('*.py'):
    shutil.copy(f, pathlib.Path(SRC_OUT) / f.name)
    copied.append(f.name)
print(f'✅ Copied {len(copied)} files: {sorted(copied)}')

In [ ]:
# ═══ Cell 2 — Install deps ═══
!pip install -q --upgrade python-chess huggingface_hub pandas numpy orjson zstandard bottleneck

In [ ]:
# ═══ Cell 3 — Patch sync.py + engine.py + load sync ═══
import os, sys, pathlib
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['HF_USER'] = 'OmarCodePlay'

# Rewrite sync.py with env precedence
sp = pathlib.Path('/kaggle/working/src/sync.py')
sp.write_text('''import os
from huggingface_hub import HfApi, hf_hub_download

TOKEN = os.environ.get("HF_TOKEN")
if not TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        TOKEN = None
if not TOKEN:
    raise RuntimeError("HF_TOKEN not found")

USER = os.environ.get("HF_USER", "YOUR_HF_USERNAME")
MODEL_REPO = f"{USER}/chess-ckpt"
DATA_REPO = f"{USER}/chess-data"
api = HfApi(token=TOKEN)

def init():
    api.create_repo(MODEL_REPO, private=True, exist_ok=True)
    api.create_repo(DATA_REPO, private=True, repo_type="dataset", exist_ok=True)

def push_ckpt(path, name=None):
    api.upload_file(path_or_fileobj=path, path_in_repo=name or os.path.basename(path), repo_id=MODEL_REPO)

def pull_ckpt(name, d, force=False):
    local = os.path.join(d, name)
    if not force and os.path.exists(local):
        return local
    try:
        return hf_hub_download(MODEL_REPO, name, local_dir=d, token=TOKEN, force_download=force)
    except Exception:
        return None

def push_data(path, name=None):
    api.upload_file(path_or_fileobj=path, path_in_repo=name or os.path.basename(path), repo_id=DATA_REPO, repo_type="dataset")

def pull_data(name, d):
    return hf_hub_download(DATA_REPO, name, repo_type="dataset", local_dir=d, token=TOKEN)

def list_data():
    return [f.rfilename for f in api.list_repo_tree(DATA_REPO, repo_type="dataset")]
''')
print('✅ sync.py rewritten')

# Patch engine.py zobrist bug
ep = pathlib.Path('/kaggle/working/src/engine.py')
if ep.exists():
    esrc = ep.read_text()
    if 'board.zobrist_hash()' in esrc:
        esrc = esrc.replace('board.zobrist_hash()', 'chess.polyglot.zobrist_hash(board)')
        if 'import chess.polyglot' not in esrc:
            esrc = esrc.replace('import chess.syzygy', 'import chess.syzygy\nimport chess.polyglot')
        ep.write_text(esrc)
        print('✅ engine.py zobrist fixed')

# Load sync
for k in list(sys.modules):
    if k == 'sync' or k.startswith('sync.'):
        del sys.modules[k]
sys.path.insert(0, '/kaggle/working/src')
import sync
print(f'✅ sync loaded, token={sync.TOKEN[:6]}...')

In [ ]:
# ═══ Cell 4 — Inspect chess-data + fail loudly if base shards missing ═══
existing = sync.list_data()
print(f'=== chess-data: {len(existing)} files ===')
for f in sorted(existing):
    print(f'  {f}')

# Base shards that MUST exist for training
critical = ['train_0000.npz', 'train_0001.npz', 'val.npz', 'test.npz']
missing_critical = [c for c in critical if c not in existing]

if missing_critical:
    print(f'\n❌ FATAL: missing base shards from chess-data:')
    for c in missing_critical:
        print(f'   {c}')
    print()
    print('   These are the game shards + val/test set built by build_data.py.')
    print('   Upload them from your local PC to chess-data:')
    print('   https://huggingface.co/datasets/OmarCodePlay/chess-data/upload/main')
    raise SystemExit('Cannot continue — base shards missing')

print(f'\n✅ All base shards present: {critical}')

# What's missing from the witty set
witty_need = ['draws_witty_0000.npz', 'resign_witty_0000.npz']
missing_witty = [w for w in witty_need if w not in existing]
print(f'\nWitty shards to add: {missing_witty if missing_witty else "none — all present"}')

In [ ]:
# ═══ Cell 5 — Auto-fetch missing shards from GM-games-with-eval -> chess-data ═══
import os, shutil
from huggingface_hub import hf_hub_download

ARCHIVE_REPO = 'OmarCodePlay/GM-games-with-eval'
os.makedirs('/kaggle/working/fetched', exist_ok=True)

# Shards we want in chess-data, source repo lookup order
# (draws_0000/resign_0000 are Hikaru; draws_witty/resign_witty are Witty_Alien)
to_fetch = [
    'draws_0000.npz',
    'resign_0000.npz',
    'draws_witty_0000.npz',
    'resign_witty_0000.npz',
]

existing_now = set(sync.list_data())
for name in to_fetch:
    if name in existing_now:
        print(f'✅ {name} already in chess-data')
        continue

    # Try to pull from archive repo
    try:
        print(f'⬇️  fetching {name} from {ARCHIVE_REPO}...')
        local = hf_hub_download(
            ARCHIVE_REPO, name,
            repo_type='dataset',
            local_dir='/kaggle/working/fetched',
            token=os.environ['HF_TOKEN'])
        size_mb = os.path.getsize(local) / 1e6
        print(f'   ✅ downloaded ({size_mb:.2f} MB)')

        # Upload to chess-data
        print(f'⬆️  pushing {name} to chess-data...')
        sync.push_data(local, name)
        print(f'   ✅ uploaded')
    except Exception as e:
        print(f'   ⚠️  could not fetch {name}: {e}')
        print(f'      → upload manually to chess-data from PC if you have it locally')

# Final state
final_files = sync.list_data()
print(f'\n=== chess-data after fetch: {len(final_files)} files ===')
for name in to_fetch:
    print(f"{'✅' if name in final_files else '❌'} {name}")

In [ ]:
# ═══ Cell 6 — REBUILD index.json from ACTUAL files on chess-data ═══
# (the previous version read the old index which was corrupted to only-puzzles)
import json

files = set(sync.list_data())

train = []

# ── Games × 3 (critical to balance against 6M puzzles) ──
for _ in range(3):
    if 'train_0000.npz' in files:
        train.append('train_0000.npz')
    if 'train_0001.npz' in files:
        train.append('train_0001.npz')

# ── All puzzle shards ──
puzzles = sorted([f for f in files if f.startswith('puzzles_') and f.endswith('.npz')])
train.extend(puzzles)

# ── Value-head shards ──
for name in ['draws_0000.npz', 'resign_0000.npz',
             'draws_witty_0000.npz', 'resign_witty_0000.npz']:
    if name in files:
        train.append(name)

idx = {
    'train': train,
    'val': ['val.npz'],
    'test': ['test.npz'],
}

with open('/tmp/index.json', 'w') as f:
    json.dump(idx, f, indent=2)

sync.push_data('/tmp/index.json', 'index.json')

print(f'✅ index.json rebuilt with {len(train)} train entries')
print()
for s in train:
    print(f'  {s}')
print()

# Verify all 4 value shards made it in
for name in ['draws_0000.npz', 'resign_0000.npz',
             'draws_witty_0000.npz', 'resign_witty_0000.npz']:
    print(f"{'✅' if name in train else '❌'} {name}")

In [ ]:
# ═══ Cell 7 — All train.py patches (v_mask + pol_loss masking + v5 config) ═══
import pathlib
p = pathlib.Path('/kaggle/working/src/train.py')
src = p.read_text()

# Fix map_location
src = src.replace('map_location=local_rank', 'map_location=f"cuda:{local_rank}"')

# ── PATCH 1: load_chunk — add v_mask (puzzles = policy-only) ──
old_load = '''def load_chunk(names, device):
    arr = {k: [] for k in KEYS}
    local = os.path.join(DATA, f"r{device}")
    os.makedirs(local, exist_ok=True)
    for n in names:
        p = os.path.join(local, n)
        if not os.path.exists(p):
            p = sync.pull_data(n, local)
        z = np.load(p)
        for k in KEYS:
            arr[k].append(z[k])
    return {k: torch.from_numpy(np.concatenate(v)).to(device) for k, v in arr.items()}'''

new_load = '''def load_chunk(names, device):
    arr = {k: [] for k in KEYS}
    vmasks = []
    local = os.path.join(DATA, f"r{device}")
    os.makedirs(local, exist_ok=True)
    for n in names:
        p = os.path.join(local, n)
        if not os.path.exists(p):
            p = sync.pull_data(n, local)
        z = np.load(p)
        for k in KEYS:
            arr[k].append(z[k])
        is_puzzle = n.startswith("puzzles_")
        vm = np.zeros(z["val"].shape[0], dtype=np.float16)
        if not is_puzzle:
            vm.fill(1.0)
        vmasks.append(vm)
    out = {k: torch.from_numpy(np.concatenate(v)).to(device) for k, v in arr.items()}
    out["vmask"] = torch.from_numpy(np.concatenate(vmasks)).to(device)
    return out'''

if old_load in src:
    src = src.replace(old_load, new_load)
    print('✅ load_chunk patched (v_mask)')
elif 'vmask' in src:
    print('✅ load_chunk already patched')
else:
    print('❌ load_chunk NOT found — cannot patch')

# ── PATCH 2: training loop uses v_mask ──
old_loop = '''                b = perm_local[i: i + B]
                bb, meta, mvk, wk, v = [data[k][b] for k in KEYS]
                for g in opt.param_groups:
                    g["lr"] = lr_at(st["step"])
                with torch.autocast("cuda", dtype=torch.float16):
                    lg, pv = ddp(bb, meta)
                lp = pol_loss(lg, mvk, wk, C["smooth"])
                lv = F.mse_loss(pv.float(), v)
                loss = lp + C["v_w"] * lv'''

new_loop = '''                b = perm_local[i: i + B]
                bb, meta, mvk, wk, v = [data[k][b] for k in KEYS]
                vmask = data["vmask"][b].float()
                for g in opt.param_groups:
                    g["lr"] = lr_at(st["step"])
                with torch.autocast("cuda", dtype=torch.float16):
                    lg, pv = ddp(bb, meta)
                lp = pol_loss(lg, mvk, wk, C["smooth"])
                lv_raw = F.mse_loss(pv.float(), v, reduction="none")
                lv = (lv_raw * vmask).sum() / vmask.sum().clamp_min(1.0)
                loss = lp + C["v_w"] * lv'''

if old_loop in src:
    src = src.replace(old_loop, new_loop)
    print('✅ training loop patched (v_mask in value loss)')
elif 'vmask = data["vmask"]' in src:
    print('✅ training loop already patched')
else:
    print('❌ training loop NOT found')

# ── PATCH 3: pol_loss masks value-only samples ──
old_pol = '''def pol_loss(lg, mvk, wk, smooth):
    lp = F.log_softmax(lg.float(), 1)
    w = wk.float()
    w = w / w.sum(1, keepdim=True).clamp_min(1e-6)
    nll = -(w * lp.gather(1, mvk.long().clamp_min(0))).sum(1).mean()
    return (1 - smooth) * nll + smooth * (-lp.mean(1)).mean()'''

new_pol = '''def pol_loss(lg, mvk, wk, smooth):
    has_policy = (wk.sum(dim=1) > 0)
    if not has_policy.any():
        return torch.zeros((), device=lg.device, dtype=torch.float32)
    lg_m = lg[has_policy]
    mvk_m = mvk[has_policy]
    wk_m = wk[has_policy]
    lp = F.log_softmax(lg_m.float(), 1)
    w = wk_m.float()
    w = w / w.sum(1, keepdim=True).clamp_min(1e-6)
    nll = -(w * lp.gather(1, mvk_m.long().clamp_min(0))).sum(1).mean()
    return (1 - smooth) * nll + smooth * (-lp.mean(1)).mean()'''

if old_pol in src:
    src = src.replace(old_pol, new_pol)
    print('✅ pol_loss masked')
elif 'has_policy' in src:
    print('✅ pol_loss already masked')
else:
    print('❌ pol_loss NOT found')

# ── PATCH 4: config -> v5 ──
for old_run in ['base2', 'base3', 'base4', 'base5', 'base6', 'base7', 'v5']:
    src = src.replace(f'run="{old_run}",', 'run="v5",')

if 'init_from="engine.pt",' not in src:
    for old_init in ['init_from=None,', 'init_from="best_base3.pt",', 'init_from="best_base2.pt",']:
        src = src.replace(old_init, 'init_from="engine.pt",')

for old in ['lr=2e-3,', 'lr=3e-5,', 'lr=4e-5,', 'lr=5e-5,']:
    src = src.replace(old, 'lr=2e-5,')
for old in ['v_w=2.0,', 'v_w=0.8,', 'v_w=0.7,', 'v_w=0.5,', 'v_w=0.3,']:
    src = src.replace(old, 'v_w=1.0,')
for old in ['max_steps=12000,', 'max_steps=14000,', 'max_steps=15000,', 'max_steps=10000,', 'max_steps=None,']:
    src = src.replace(old, 'max_steps=25000,')
for old in ['decay_start=9000,', 'decay_start=10000,', 'decay_start=11000,', 'decay_start=7500,', 'decay_start=None,']:
    src = src.replace(old, 'decay_start=20000,')
for old in ['decay_steps=4000,', 'decay_steps=3000,', 'decay_steps=2500,', 'decay_steps=100_000,']:
    src = src.replace(old, 'decay_steps=5000,')
if 'patience' not in src:
    src = src.replace('max_steps=25000,', 'max_steps=25000,\n    patience=3,\n    valmse_tolerance=0.0005,')

p.write_text(src)
print('\n✅ train.py configured for v5')
print('=' * 60)
!grep -nE '^    run=|^    init_from=|^    lr=|^    v_w=|^    max_steps=|^    patience=' /kaggle/working/src/train.py

In [ ]:
# ═══ Cell 8 — Verify index + patches, then launch ═══
import os, sys, json, subprocess
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['HF_USER'] = 'OmarCodePlay'
for k in list(sys.modules):
    if k == 'sync' or k.startswith('sync.'):
        del sys.modules[k]
sys.path.insert(0, '/kaggle/working/src')
import sync

idx = json.load(open(sync.pull_data('index.json', '/kaggle/working/shards')))
print(f'Train shards ({len(idx["train"])}):')
for s in idx['train']:
    print(f'  {s}')
print(f'Val:  {idx["val"]}')
print(f'Test: {idx["test"]}\n')

# Sanity checks
checks = ['train_0000.npz', 'train_0001.npz',
          'draws_0000.npz', 'resign_0000.npz',
          'draws_witty_0000.npz', 'resign_witty_0000.npz']
for c in checks:
    print(f"{'✅' if c in idx['train'] else '❌'} {c}")

r = subprocess.run(['grep', '-c', 'vmask', '/kaggle/working/src/train.py'],
                   capture_output=True, text=True)
vmask_count = int(r.stdout.strip() or 0)
print(f'\nvmask occurrences in train.py: {vmask_count} (need ≥3)')

r = subprocess.run(['grep', '-c', 'has_policy', '/kaggle/working/src/train.py'],
                   capture_output=True, text=True)
policy_count = int(r.stdout.strip() or 0)
print(f'has_policy occurrences: {policy_count} (need ≥2)')

import torch
ck = sync.pull_ckpt('engine.pt', '/kaggle/working/ckpt')
print(f"engine.pt: {'✅ ' + str(round(os.path.getsize(ck)/1e6, 1)) + ' MB' if ck else '❌ missing'}")

print('=' * 60)
if len(idx['train']) >= 20 and vmask_count >= 3 and policy_count >= 2 and ck:
    print('✅ All checks passed — launching training')
    !cd /kaggle/working && torchrun --standalone --nproc_per_node=2 src/train.py 2>&1 | tee -a /kaggle/working/train_v5.log
else:
    print('❌ One or more checks failed — NOT launching')
    print('   Fix the ❌ items above before re-running this cell')

In [ ]:
# ═══ Cell 9 — Post-training tests (run after SESSION END) ═══
import chess, pathlib, sys
sys.path.insert(0, '/kaggle/working/src')

ep = pathlib.Path('/kaggle/working/src/engine.py')
esrc = ep.read_text()
if 'board.zobrist_hash()' in esrc:
    esrc = esrc.replace('board.zobrist_hash()', 'chess.polyglot.zobrist_hash(board)')
    if 'import chess.polyglot' not in esrc:
        esrc = esrc.replace('import chess.syzygy', 'import chess.syzygy\nimport chess.polyglot')
    ep.write_text(esrc)

from engine import Engine

print('=' * 60)
print('TEST 1: Mate-in-1 (back rank) — v5 should play mate')
print('=' * 60)
fen = '6k1/5ppp/8/8/8/8/5PPP/R5K1 w - - 0 1'
board = chess.Board(fen)
v5 = Engine('/kaggle/working/ckpt/best_v5.pt')
v3 = Engine('/kaggle/working/ckpt/engine.pt')
mv5, _ = v5.search(board, float('inf'), float('inf'), 500, None)
mv3, _ = v3.search(board, float('inf'), float('inf'), 500, None)
bc = board.copy(); bc.push(mv5)
print(f'v5: {mv5.uci()} -> {"MATE ✅" if bc.is_checkmate() else "not mate ❌"}')
print(f'v3: {mv3.uci()}')

print('\n' + '=' * 60)
print('TEST 2: Drawn position — should be near 0')
print('=' * 60)
bd = chess.Board('8/8/4k3/8/8/4K3/8/R6r w - - 0 1')
_, r5 = v5.search(bd, float('inf'), float('inf'), 200, None)
_, r3 = v3.search(bd, float('inf'), float('inf'), 200, None)
val5 = r5.w / r5.n if r5 and r5.n else 0
val3 = r3.w / r3.n if r3 and r3.n else 0
print(f'v5: {val5:+.3f}  {"✅ near 0" if abs(val5) < 0.3 else "⚠️ overconfident"}')
print(f'v3: {val3:+.3f}  {"✅ near 0" if abs(val3) < 0.3 else "⚠️ overconfident"}')

print('\n' + '=' * 60)
print('TEST 3: Won position — should be near +1')
print('=' * 60)
bw = chess.Board('rnb1kbnr/pppp1ppp/8/8/8/8/PPPP1PPP/RNBQKBNR w KQkq - 0 1')
_, r5w = v5.search(bw, float('inf'), float('inf'), 200, None)
_, r3w = v3.search(bw, float('inf'), float('inf'), 200, None)
val5w = r5w.w / r5w.n if r5w and r5w.n else 0
val3w = r3w.w / r3w.n if r3w and r3w.n else 0
print(f'v5: {val5w:+.3f}  {"✅ near +1" if val5w > 0.5 else "⚠️ weak"}')
print(f'v3: {val3w:+.3f}  {"✅ near +1" if val3w > 0.5 else "⚠️ weak"}')